# Datamine BLKTRI Process Example

This notebook demonstrates how to configure and run the **`blktri`** process wrapper in `dmstudio`.

## Process Description

## BLKTRI

Converts block model(s) to wireframe surfaces(s).

Although it does not support a keyfield option, the [MODTRI](<modtri.md>) process is quicker to execute than the **BLKTRI** process. Also, **MODTRI** is able to process larger models than **BLKTRI**.

Another option is to make use of the Isosurfaces wireframe function, resulting in surface shapes representing real surfaces - although note that the output from **BLKTRI** and Isosurface generation may not be identical due to the calculation methods used.

### Input Files:

* **in** (Block Model):
  Input model file. Must contain fields XC, YC, ZC, XINC, YINC, ZINC, XMORIG, YMORIG,
  ZMORIG, NX, NY, NZ, and IJK. If it is a Rotated Model then it must also include the
  fields X0, Y0, Z0, ANGLE1, ANGLE2, ANGLE3, ROTAXIS1, ROTAXIS2, and ROTAXIS3.
  Required=Yes

### Output Files:

* **wiretr** (Wireframe Triangle):
  Output wireframe triangle file.
  Required=Yes

* **wirept** (Wireframe Points):
  Output wireframe point file.
  Required=Yes

### Fields:

* **class** (Any : IN):
  Field in block model defining multiple zones or seams.
  Default=Undefined
  Required=No

* **modcol** (Numeric : IN):
  A numeric field to be used to allocate (an integer) wireframe colour. It is assumed
  that colour is related to CLASS. If colour varies within a CLASS then the colour
  corresponding to the first occurrence of each CLASS will be used.
  Default=Undefined
  Required=No

### Parameters:

* **plane**:
  Plane for interpretation of solid or seam orientation. Values are: 0 - solid model, so
  plane not required. 1 - XY plane (plan) 2 - XZ plane (East-West section) 3 - YZ plane
  (North-South section)
  Range=0,3
  Values=0,1,2,3
  Default=0
  Required=Yes

* **xsubcell**:
  Cell division in X direction.
  Range=1,20
  Values=1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
  Default=1
  Required=Yes

* **ysubcell**:
  Cell division in Y direction.
  Range=1,20
  Values=1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
  Default=1
  Required=Yes

* **zsubcell**:
  Cell division in Z direction.
  Range=1,20
  Values=1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
  Default=1
  Required=Yes

* **order**:
  If non-zero then the process treats values of the CLASS field as an ordered numeric
  sequence, and infers intermediate values to generate a sequence of wireframe seam
  surfaces. A value of -1 indicates that the numeric sequence increases in value with
  depth, and +1 a decrease with depth.
  Range=-1,1
  Values=-1,0,1
  Default=0
  Required=No

* **surface**:
  This parameter is used to specify which wireframe surfaces are created: 1 - outer
  surface (Solid model only) 2 - inner surface (Solid model only) 3 - outer and inner
  surfaces (Solid model only) 4 - bottom surface 5 -top surface 6 - both bottom and top
  surfaces All options apply to a solid model (PLANE=0), but only 4,5 and 6 apply to seams

## (PLANE=1,2,3).

  Range=1,6
  Values=1,2,3,4,5,6
  Default=3
  Required=Yes

* **colour**:
  Colour for output wireframe. Only used if a colour field is not specified for MODCOL.
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('blktri')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute blktri
print("Running blktri...")
dm_cmd.blktri(
    in_i='t_assays',  # required
    wiretr_o='t_blktri_out',  # required
    wirept_o='t_blktri_out',  # required
    plane_p='required_val',  # required
    xsubcell_p='required_val',  # required
    ysubcell_p='required_val',  # required
    zsubcell_p='required_val',  # required
    surface_p='required_val',  # required
    # class_f=['optional_field'],  # optional
    # modcol_f='optional',  # optional
    # order_p=0,  # optional
    # colour_p=1,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("blktri execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_blktri_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")